# Task 7b: Cluster Label Validation

This notebook performs quantitative validation of the refined cluster assignments (`final_cluster`):
1. **Label Term Coverage**:
   - Pattern matching on unmasked text across clusters 0 to 6 using whole words and stem wildcards.
   - Reporting cluster size $n$, coverage (share of cluster matching $\ge 1$ pattern), coverage per pattern, precision across the full 158-country corpus, and inventory of non-matching members.
2. **Opening Formula Alignment**:
   - Structural mapping of preamble opening types (`we_the_people` $\to$ Cluster 0, `whereas` $\to$ Cluster 4, `in_the_name_of` $\to$ Cluster 5).
   - In-cluster share, corpus-wide concentration in target cluster, and 7-cluster distribution.
3. **Cluster Cohesion (Silhouette Analysis)**:
   - Cosine silhouette sample scores evaluated on centered masked embeddings ($C_c$).
   - Cluster-level mean and median, inventory of countries with negative silhouette values, and overall dataset mean.
4. **Geographic Region Baseline**:
   - Resolving missing continent assignments using Natural Earth 50m cultural vectors (`ne_50m_admin_0_countries.zip`).
   - Reporting Adjusted Rand Index (ARI), Normalized Mutual Information (NMI), leave-one-out majority-vote accuracy, largest-cluster baseline accuracy, and the continent contingency matrix.
5. **Within-Region Splits**:
   - Inventory of countries per continent grouped by assigned final cluster.
6. **Validation Summary Export**:
   - Persisting `data/processed/label_validation.csv`.


## 1. Input Verification and Centered Masked Embeddings ($C_c$)

Load the refined clustering table, unmasked preamble texts, country ordering, and masked embeddings.
Verify alignment across all 158 rows and confirm cluster sizes match `[30, 26, 25, 23, 22, 17, 15]`.

In [1]:
import os
import re
import numpy as np
import pandas as pd
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_samples,
    silhouette_score
)
import geopandas as gpd

data_dir = '../data/processed'

# 1. Load inputs
df_final = pd.read_csv(os.path.join(data_dir, 'cluster_final.csv'))
preamble_df = pd.read_csv(os.path.join(data_dir, 'preamble_final.csv'))
order_df = pd.read_csv(os.path.join(data_dir, 'final_country_order.csv'))
emb_masked = np.load(os.path.join(data_dir, 'emb_final_masked.npy'))

# 2. Assertions
assert len(df_final) == 158, f"Expected 158 rows in cluster_final.csv, got {len(df_final)}"
assert len(order_df) == 158, f"Expected 158 rows in order file, got {len(order_df)}"
assert (df_final['Country'] == order_df['Country']).all(), "Country alignment mismatch between cluster_final and order file!"

clusterable_preambles = preamble_df[preamble_df['status'].isin(['preamble_ok', 'preamble_cut'])].copy().reset_index(drop=True)
assert len(clusterable_preambles) == 158, f"Expected 158 clusterable preambles, got {len(clusterable_preambles)}"
assert (df_final['Country'] == clusterable_preambles['Country']).all(), "Country alignment mismatch between cluster_final and preamble_final!"

# Check cluster sizes [30, 26, 25, 23, 22, 17, 15]
final_sizes = df_final['final_cluster'].value_counts().tolist()
assert final_sizes == [30, 26, 25, 23, 22, 17, 15], f"Expected sizes [30, 26, 25, 23, 22, 17, 15], got {final_sizes}"
print(f"Validated inputs: 158 aligned countries. Final cluster sizes: {final_sizes}")

# 3. Construct C_c (subtract column mean, L2-normalize rows)
emb_C_c = emb_masked - emb_masked.mean(axis=0, keepdims=True)
emb_C_c = emb_C_c / np.linalg.norm(emb_C_c, axis=1, keepdims=True)
print(f"Constructed C_c with shape {emb_C_c.shape}, unit row norms verified.")


Validated inputs: 158 aligned countries. Final cluster sizes: [30, 26, 25, 23, 22, 17, 15]
Constructed C_c with shape (158, 768), unit row norms verified.


## 2. Label Term Coverage on Unmasked Extracted Text

Evaluate pattern matching on unmasked preamble text (`extracted_text`) using case-insensitive whole-word and prefix-stem rules:
- **Cluster 0**: `justice`, `freedom`, `equality`
- **Cluster 1**: `african*`, `charter`, `declaration`, `universal`
- **Cluster 2**: `statehood`, `century`, `centuries`, `historical`, `responsib*`
- **Cluster 3**: `invok*`, `decree*`, `constituent`, `assembly`
- **Cluster 4**: `supremacy`, `recognit*`, `whereas`, `effect`
- **Cluster 5**: `merciful`, `compassionate`, `islam*`, `arab*`
- **Cluster 6**: `socialis*`, `party`, `parties`, `struggle*`, `independence`

Metrics computed per cluster:
- **$n$**: Cluster size.
- **Coverage**: Fraction of cluster members matching at least one pattern.
- **Coverage per pattern**: Frequency and share of cluster members matching each individual pattern.
- **Precision**: Number of matching cluster members divided by all 158 countries matching $\ge 1$ pattern.
- **Non-matching members**: Complete list of cluster members matching none of the patterns.

In [2]:
patterns_raw = {
    0: ['justice', 'freedom', 'equality'],
    1: ['african*', 'charter', 'declaration', 'universal'],
    2: ['statehood', 'century', 'centuries', 'historical', 'responsib*'],
    3: ['invok*', 'decree*', 'constituent', 'assembly'],
    4: ['supremacy', 'recognit*', 'whereas', 'effect'],
    5: ['merciful', 'compassionate', 'islam*', 'arab*'],
    6: ['socialis*', 'party', 'parties', 'struggle*', 'independence']
}

def build_regex(p):
    if p.endswith('*'):
        stem = p[:-1]
        return re.compile(r'\b' + re.escape(stem) + r'\w*\b', re.IGNORECASE)
    else:
        return re.compile(r'\b' + re.escape(p) + r'\b', re.IGNORECASE)

patterns = {c: [(p, build_regex(p)) for p in pat_list] for c, pat_list in patterns_raw.items()}

coverage_summary = []
non_matching_by_cluster = {}

print("=== LABEL TERM COVERAGE AND PRECISION PER CLUSTER ===")
for c in range(7):
    pats = patterns[c]
    sub_c = df_final[df_final['final_cluster'] == c]
    n_c = len(sub_c)
    
    c_matches = []
    all_matches = []
    pat_counts_in_cluster = {p: 0 for p, _ in pats}
    none_members = []
    
    for idx, r in df_final.iterrows():
        text = str(clusterable_preambles.loc[idx, 'extracted_text'])
        m_any = False
        for p_str, p_regex in pats:
            if p_regex.search(text):
                m_any = True
                if r['final_cluster'] == c:
                    pat_counts_in_cluster[p_str] += 1
        if m_any:
            all_matches.append(r['Country'])
            if r['final_cluster'] == c:
                c_matches.append(r['Country'])
        else:
            if r['final_cluster'] == c:
                none_members.append(r['Country'])
                
    cov = len(c_matches) / n_c
    prec = len(c_matches) / len(all_matches) if len(all_matches) > 0 else 0.0
    non_matching_by_cluster[c] = none_members
    
    coverage_summary.append({
        'cluster': c,
        'n': n_c,
        'coverage': cov,
        'precision': prec,
        'member_matches': len(c_matches),
        'total_matches': len(all_matches)
    })
    
    print(f"\n--- Cluster {c} (n={n_c}) ---")
    print(f"  Coverage:  {len(c_matches)}/{n_c} ({cov:.4f} or {cov*100:.2f}%)")
    print(f"  Precision: {len(c_matches)}/{len(all_matches)} ({prec:.4f} or {prec*100:.2f}%)")
    print(f"  Coverage per pattern:")
    for p_str, _ in pats:
        cnt = pat_counts_in_cluster[p_str]
        print(f"    {p_str:<15}: {cnt:>2}/{n_c} ({cnt/n_c:.4f} or {cnt/n_c*100:.2f}%)")
    print(f"  Members matching none ({len(none_members)}): {none_members}")


=== LABEL TERM COVERAGE AND PRECISION PER CLUSTER ===

--- Cluster 0 (n=30) ---
  Coverage:  24/30 (0.8000 or 80.00%)
  Precision: 24/129 (0.1860 or 18.60%)
  Coverage per pattern:
    justice        : 19/30 (0.6333 or 63.33%)
    freedom        : 16/30 (0.5333 or 53.33%)
    equality       : 14/30 (0.4667 or 46.67%)
  Members matching none (6): ['Cambodia', 'Costa Rica', 'Malawi', 'Nauru', 'Taiwan', 'Turkmenistan']

--- Cluster 1 (n=26) ---
  Coverage:  21/26 (0.8077 or 80.77%)
  Precision: 21/57 (0.3684 or 36.84%)
  Coverage per pattern:
    african*       : 21/26 (0.8077 or 80.77%)
    charter        : 17/26 (0.6538 or 65.38%)
    declaration    : 18/26 (0.6923 or 69.23%)
    universal      : 18/26 (0.6923 or 69.23%)
  Members matching none (5): ['Eritrea', 'Gambia', 'Mozambique', 'Namibia', 'Sudan']

--- Cluster 2 (n=25) ---
  Coverage:  21/25 (0.8400 or 84.00%)
  Precision: 21/66 (0.3182 or 31.82%)
  Coverage per pattern:
    statehood      : 12/25 (0.4800 or 48.00%)
    century  

## 3. Opening Formula Alignment (Form versus Content)

Using the `opening_type` classification from `cluster_final.csv`, evaluate the concentration of characteristic opening formulas in their primary clusters:
- `we_the_people` $\to$ Cluster 0
- `whereas` $\to$ Cluster 4
- `in_the_name_of` $\to$ Cluster 5

For each formula, report:
1. Share of target cluster members opening with that formula.
2. Share of all corpus countries opening with that formula that sit in the target cluster.
3. Distribution of the opening formula across all 7 clusters.

In [3]:
target_openings = {
    'we_the_people': 0,
    'whereas': 4,
    'in_the_name_of': 5
}

print("=== OPENING FORMULA DISTRIBUTION AND TARGET CLUSTER ALIGNMENT ===")
for op, target_cluster in target_openings.items():
    sub_c = df_final[df_final['final_cluster'] == target_cluster]
    n_c = len(sub_c)
    n_c_op = (sub_c['opening_type'] == op).sum()
    share_of_cluster = n_c_op / n_c
    
    all_op = df_final[df_final['opening_type'] == op]
    tot_op = len(all_op)
    share_of_all = n_c_op / tot_op
    
    dist = df_final[df_final['opening_type'] == op]['final_cluster'].value_counts().reindex(range(7), fill_value=0)
    
    print(f"\n--- Opening: '{op}' (Target Cluster: {target_cluster}) ---")
    print(f"  Cluster member share:   {n_c_op}/{n_c} ({share_of_cluster:.4f} or {share_of_cluster*100:.2f}%)")
    print(f"  Total formula share:    {n_c_op}/{tot_op} ({share_of_all:.4f} or {share_of_all*100:.2f}%)")
    print(f"  Distribution across clusters 0..6:")
    for c in range(7):
        pct = dist[c] / tot_op * 100
        print(f"    Cluster {c}: {dist[c]:>2} ({pct:>5.1f}%)")


=== OPENING FORMULA DISTRIBUTION AND TARGET CLUSTER ALIGNMENT ===

--- Opening: 'we_the_people' (Target Cluster: 0) ---
  Cluster member share:   22/30 (0.7333 or 73.33%)
  Total formula share:    22/62 (0.3548 or 35.48%)
  Distribution across clusters 0..6:
    Cluster 0: 22 ( 35.5%)
    Cluster 1: 11 ( 17.7%)
    Cluster 2:  9 ( 14.5%)
    Cluster 3: 10 ( 16.1%)
    Cluster 4:  5 (  8.1%)
    Cluster 5:  0 (  0.0%)
    Cluster 6:  5 (  8.1%)

--- Opening: 'whereas' (Target Cluster: 4) ---
  Cluster member share:   12/22 (0.5455 or 54.55%)
  Total formula share:    12/18 (0.6667 or 66.67%)
  Distribution across clusters 0..6:
    Cluster 0:  3 ( 16.7%)
    Cluster 1:  1 (  5.6%)
    Cluster 2:  0 (  0.0%)
    Cluster 3:  0 (  0.0%)
    Cluster 4: 12 ( 66.7%)
    Cluster 5:  1 (  5.6%)
    Cluster 6:  1 (  5.6%)

--- Opening: 'in_the_name_of' (Target Cluster: 5) ---
  Cluster member share:   10/17 (0.5882 or 58.82%)
  Total formula share:    10/16 (0.6250 or 62.50%)
  Distribution acro

## 4. Cluster Cohesion (Cosine Silhouette Analysis)

Compute individual silhouette sample scores using cosine metric on the centered masked representations ($C_c$) against `final_cluster`.
Report:
- Mean and median silhouette per cluster.
- Number and member list of preambles with negative silhouette scores.
- Overall dataset mean silhouette score.

In [4]:
sil_samples = silhouette_samples(emb_C_c, df_final['final_cluster'].values, metric='cosine')
df_final['silhouette'] = sil_samples
overall_mean_sil = sil_samples.mean()

print(f"=== CLUSTER COHESION (COSINE SILHOUETTE) ===")
print(f"Overall Mean Silhouette: {overall_mean_sil:.4f}\n")

cohesion_records = []
all_neg_countries = []

for c in range(7):
    sub = df_final[df_final['final_cluster'] == c]
    mean_sil = sub['silhouette'].mean()
    median_sil = sub['silhouette'].median()
    neg_sub = sub[sub['silhouette'] < 0]
    neg_countries = neg_sub['Country'].tolist()
    all_neg_countries.extend(neg_countries)
    
    cohesion_records.append({
        'cluster': c,
        'mean_silhouette': mean_sil,
        'median_silhouette': median_sil,
        'negative_count': len(neg_sub),
        'negative_members': neg_countries
    })
    
    print(f"--- Cluster {c} (n={len(sub)}) ---")
    print(f"  Mean Silhouette:   {mean_sil:.4f}")
    print(f"  Median Silhouette: {median_sil:.4f}")
    print(f"  Negative Members ({len(neg_sub)}): {neg_countries}")

print(f"\nTotal countries with negative silhouette: {len(all_neg_countries)}")


=== CLUSTER COHESION (COSINE SILHOUETTE) ===
Overall Mean Silhouette: 0.1606

--- Cluster 0 (n=30) ---
  Mean Silhouette:   0.1321
  Median Silhouette: 0.1274
  Negative Members (2): ['Cambodia', 'Ethiopia']
--- Cluster 1 (n=26) ---
  Mean Silhouette:   0.2127
  Median Silhouette: 0.2133
  Negative Members (0): []
--- Cluster 2 (n=25) ---
  Mean Silhouette:   0.1768
  Median Silhouette: 0.1920
  Negative Members (0): []
--- Cluster 3 (n=23) ---
  Mean Silhouette:   0.1625
  Median Silhouette: 0.1629
  Negative Members (0): []
--- Cluster 4 (n=22) ---
  Mean Silhouette:   0.1142
  Median Silhouette: 0.0835
  Negative Members (1): ['Saint Lucia']
--- Cluster 5 (n=17) ---
  Mean Silhouette:   0.2007
  Median Silhouette: 0.2298
  Negative Members (0): []
--- Cluster 6 (n=15) ---
  Mean Silhouette:   0.1198
  Median Silhouette: 0.1410
  Negative Members (1): ['S. Sudan']

Total countries with negative silhouette: 4


## 5. Geographic Region Baseline (Natural Earth 50m Cultural Vectors)

Supplement blank continent values in `cluster_final.csv` by matching against Natural Earth 50m cultural vectors (`ne_50m_admin_0_countries.zip`).
Evaluate:
- Remaining unmatched countries.
- Adjusted Rand Index (ARI) and Normalized Mutual Information (NMI) between `final_cluster` and `continent`.
- Leave-one-out majority-vote prediction accuracy.
- Baseline accuracy of always predicting the largest cluster.
- Continent-by-cluster contingency matrix.

In [5]:
# 1. Load 50m cultural vectors
ne_50m_path = '/tmp/ne_50m_admin_0_countries.zip'
if not os.path.exists(ne_50m_path):
    world_50m = gpd.read_file("https://naturalearth.s3.amazonaws.com/50m_cultural/ne_50m_admin_0_countries.zip")
else:
    world_50m = gpd.read_file(ne_50m_path)

print(f"Loaded Natural Earth 50m dataset with {len(world_50m)} features.")

# Fill blank continents via exact NAME match
name_to_cont_50m = dict(zip(world_50m['NAME'], world_50m['CONTINENT']))
filled_continent = df_final['continent'].copy()

newly_matched = []
for i, r in df_final.iterrows():
    if pd.isna(r['continent']):
        c_name = r['Country']
        if c_name in name_to_cont_50m and pd.notna(name_to_cont_50m[c_name]):
            filled_continent.iloc[i] = name_to_cont_50m[c_name]
            newly_matched.append((c_name, name_to_cont_50m[c_name]))

print(f"Newly matched from 50m data ({len(newly_matched)}):")
for c_name, cont in newly_matched:
    print(f"  {c_name:<25}: {cont}")

df_final['continent_filled'] = filled_continent
still_unmatched = df_final[df_final['continent_filled'].isna()]['Country'].tolist()
print(f"\nStill unmatched countries ({len(still_unmatched)}): {still_unmatched}")

# Subset to matched countries
matched_geo = df_final.dropna(subset=['continent_filled']).copy().reset_index(drop=True)
n_geo = len(matched_geo)
print(f"\nTotal matched countries: {n_geo} / 158")

# 2. ARI and NMI
ari_geo = adjusted_rand_score(matched_geo['final_cluster'], matched_geo['continent_filled'])
nmi_geo = normalized_mutual_info_score(matched_geo['final_cluster'], matched_geo['continent_filled'])
print(f"\nARI(final_cluster, continent): {ari_geo:.4f}")
print(f"NMI(final_cluster, continent): {nmi_geo:.4f}")

# 3. Leave-one-out majority-vote accuracy
loo_correct = 0
for i in range(n_geo):
    own_c = matched_geo.loc[i, 'final_cluster']
    own_cont = matched_geo.loc[i, 'continent_filled']
    others = matched_geo[(matched_geo['continent_filled'] == own_cont) & (matched_geo.index != i)]
    if len(others) == 0:
        continue
    counts = others['final_cluster'].value_counts()
    pred_c = counts.index[0] # mode of continent members
    if pred_c == own_c:
        loo_correct += 1

loo_acc = loo_correct / n_geo
print(f"Leave-one-out majority-vote accuracy: {loo_correct}/{n_geo} ({loo_acc:.4f} or {loo_acc*100:.2f}%)")

# 4. Always predicting largest cluster accuracy
largest_c = matched_geo['final_cluster'].value_counts().idxmax()
largest_cnt = (matched_geo['final_cluster'] == largest_c).sum()
largest_acc = largest_cnt / n_geo
print(f"Always predicting largest cluster ({largest_c}) accuracy: {largest_cnt}/{n_geo} ({largest_acc:.4f} or {largest_acc*100:.2f}%)")

# 5. Crosstab
ct_geo = pd.crosstab(matched_geo['final_cluster'], matched_geo['continent_filled'])
print(f"\n=== CROSSTAB: FINAL CLUSTER BY CONTINENT (MATCHED n={n_geo}) ===")
print(ct_geo)


Loaded Natural Earth 50m dataset with 242 features.
Newly matched from 50m data (15):
  Andorra                  : Europe
  Bahrain                  : Asia
  Barbados                 : North America
  Cabo Verde               : Africa
  Comoros                  : Africa
  Dominica                 : North America
  Grenada                  : North America
  Kiribati                 : Oceania
  Micronesia               : Oceania
  Nauru                    : Oceania
  Saint Lucia              : North America
  Samoa                    : Oceania
  São Tomé and Principe    : Africa
  Seychelles               : Seven seas (open ocean)
  Tuvalu                   : Oceania

Still unmatched countries (3): ['Antigua and Barbuda', 'Marshall Islands', 'Saint Kitts and Nevis']

Total matched countries: 155 / 158

ARI(final_cluster, continent): 0.3013
NMI(final_cluster, continent): 0.4503
Leave-one-out majority-vote accuracy: 71/155 (0.4581 or 45.81%)
Always predicting largest cluster (0) accuracy: 

## 6. Within-Region Splits

Inventory each continent's countries grouped by assigned `final_cluster`.

In [6]:
print("=== WITHIN-REGION CLUSTER SPLITS ===")
for cont in sorted(matched_geo['continent_filled'].unique()):
    sub_cont = matched_geo[matched_geo['continent_filled'] == cont]
    print(f"\n=== Continent: {cont} (n={len(sub_cont)}) ===")
    for c in sorted(sub_cont['final_cluster'].unique()):
        cnt_list = sorted(sub_cont[sub_cont['final_cluster'] == c]['Country'].tolist())
        print(f"  Cluster {c} ({len(cnt_list):>2}): {cnt_list}")


=== WITHIN-REGION CLUSTER SPLITS ===

=== Continent: Africa (n=48) ===
  Cluster 0 (11): ['Ethiopia', 'Ghana', 'Kenya', 'Liberia', 'Malawi', 'South Africa', 'Tanzania', 'Uganda', 'Zambia', 'Zimbabwe', 'eSwatini']
  Cluster 1 (26): ['Angola', 'Benin', 'Burkina Faso', 'Burundi', 'Cameroon', 'Central African Rep.', 'Chad', 'Congo', "Côte d'Ivoire", 'Dem. Rep. Congo', 'Djibouti', 'Eq. Guinea', 'Eritrea', 'Gabon', 'Gambia', 'Guinea', 'Mali', 'Mauritania', 'Mozambique', 'Namibia', 'Niger', 'Nigeria', 'Rwanda', 'Senegal', 'Sudan', 'Togo']
  Cluster 3 ( 4): ['Cabo Verde', 'Guinea-Bissau', 'Madagascar', 'São Tomé and Principe']
  Cluster 4 ( 1): ['Comoros']
  Cluster 5 ( 5): ['Algeria', 'Egypt', 'Libya', 'Morocco', 'Tunisia']
  Cluster 6 ( 1): ['S. Sudan']

=== Continent: Asia (n=39) ===
  Cluster 0 ( 8): ['Bhutan', 'Cambodia', 'India', 'Mongolia', 'Philippines', 'Taiwan', 'Tajikistan', 'Turkmenistan']
  Cluster 2 ( 5): ['Armenia', 'Georgia', 'Kazakhstan', 'Kyrgyzstan', 'Uzbekistan']
  Cluster 

## 7. Deliverables & Validation Table Export

Export validation summary table to `data/processed/label_validation.csv` with fields:
`cluster`, `n`, `coverage`, `precision`, `mean_silhouette`, `negative_silhouette_count`, `mean_conf_sub`.

In [7]:
val_records = []
for c in range(7):
    cov_info = coverage_summary[c]
    coh_info = cohesion_records[c]
    mean_conf_sub_val = df_final[df_final['final_cluster'] == c]['conf_sub'].mean()
    
    val_records.append({
        'cluster': c,
        'n': cov_info['n'],
        'coverage': cov_info['coverage'],
        'precision': cov_info['precision'],
        'mean_silhouette': coh_info['mean_silhouette'],
        'negative_silhouette_count': coh_info['negative_count'],
        'mean_conf_sub': mean_conf_sub_val
    })

df_label_val = pd.DataFrame(val_records)
out_csv_path = os.path.join(data_dir, 'label_validation.csv')
df_label_val.to_csv(out_csv_path, index=False)
print(f"Saved {out_csv_path} with shape {df_label_val.shape}")

print("\n=== data/processed/label_validation.csv ===")
print(df_label_val.to_string(index=False))


Saved ../data/processed/label_validation.csv with shape (7, 7)

=== data/processed/label_validation.csv ===
 cluster  n  coverage  precision  mean_silhouette  negative_silhouette_count  mean_conf_sub
       0 30  0.800000   0.186047         0.132061                          2       0.694530
       1 26  0.807692   0.368421         0.212719                          0       0.773118
       2 25  0.840000   0.318182         0.176798                          0       0.809904
       3 23  0.869565   0.350877         0.162486                          0       0.802271
       4 22  0.727273   0.410256         0.114158                          1       0.627348
       5 17  0.941176   0.888889         0.200743                          0       0.855500
       6 15  0.933333   0.160920         0.119820                          1       0.658775
